In [89]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', None)

In [90]:
df = pd.read_csv("../data/taylor_swift_setlists.csv", parse_dates=["event_date"])
tracklists_df = pd.read_csv("../data/taylor_swift_album_tracklists.csv")

In [91]:
tour_counts = df.groupby("tour")["event_date"].agg(
    num_shows=lambda x: x.nunique(),
    first_show="min",
    last_show="max"
).sort_values("num_shows", ascending=False)

print(tour_counts)

                            num_shows first_show  last_show
tour                                                       
The Eras Tour                     149 2023-03-17 2024-12-08
Fearless                          112 2009-04-23 2010-07-10
Speak Now World Tour              111 2011-02-09 2012-03-18
The Red Tour                       86 2013-03-13 2014-06-12
The 1989 World Tour                85 2015-05-05 2015-12-12
reputation Stadium Tour            53 2018-05-08 2018-11-21
Bonfires & Amplifiers Tour         29 2007-04-26 2007-11-16
Me and My Gang Tour                 8 2006-10-19 2006-11-03
Soul2Soul II Tour                   7 2007-07-09 2007-07-18
Fearless Promo                      5 2009-03-05 2009-03-12
1989 Promo                          1 2015-12-03 2015-12-03


In [92]:
print(df["tour"].unique())

<StringArray>
[                         nan,              'The Eras Tour',
    'reputation Stadium Tour',        'The 1989 World Tour',
                 '1989 Promo',               'The Red Tour',
       'Speak Now World Tour',                   'Fearless',
             'Fearless Promo', 'Bonfires & Amplifiers Tour',
          'Soul2Soul II Tour',        'Me and My Gang Tour']
Length: 12, dtype: str


In [93]:
song_counts = df["song_name"].value_counts()
song_counts.head(20)

song_name
Love Story                                 730
You Belong With Me                         532
I Knew You Were Trouble                    436
Our Song                                   433
Fearless                                   416
We Are Never Ever Getting Back Together    368
Shake It Off                               341
Tim McGraw                                 317
Blank Space                                315
Teardrops on My Guitar                     311
Picture to Burn                            310
Should've Said No                          303
Enchanted                                  269
Fifteen                                    266
Look What You Made Me Do                   261
All Too Well                               250
22                                         248
Style                                      243
Bad Blood                                  236
Wildest Dreams                             226
Name: count, dtype: int64

In [94]:
print(f"Total unique songs: {df['song_name'].nunique()}")
print(f"Date range: {df['event_date'].min()} to {df['event_date'].max()}")

Total unique songs: 549
Date range: 2001-09-22 00:00:00 to 2026-06-09 00:00:00


In [95]:
clean_tours = [
    "Fearless", "Speak Now World Tour", "The Red Tour",
    "The 1989 World Tour", "reputation Stadium Tour", "The Eras Tour"
]

tour_df = df[df["tour"].isin(clean_tours)].copy()
print(tour_df["tour"].value_counts())

tour
The Eras Tour              7198
The Red Tour               1975
Speak Now World Tour       1952
Fearless                   1793
The 1989 World Tour        1487
reputation Stadium Tour    1231
Name: count, dtype: int64


In [96]:
tour_to_album = {
    "Fearless": "Fearless",
    "Speak Now World Tour": "Speak Now",
    "The Red Tour": "Red",
    "The 1989 World Tour": "1989",
    "reputation Stadium Tour": "reputation",
    "The Eras Tour": "Midnights"
}
tour_df["supporting_album"] = tour_df["tour"].map(tour_to_album)

tour_type = {
    "Fearless": "single_album",
    "Speak Now World Tour": "single_album",
    "The Red Tour": "single_album",
    "The 1989 World Tour": "single_album",
    "reputation Stadium Tour": "single_album",
    "The Eras Tour": "retrospective"
}
tour_df["tour_type"] = tour_df["tour"].map(tour_type)

In [97]:
album_songs_lookup = tracklists_df.groupby("album")["track_name"].apply(set).to_dict()

tour_df["is_new_album_song"] = tour_df.apply(
    lambda row: row["song_name"] in album_songs_lookup.get(row["supporting_album"], set()),
    axis=1
)

tour_df.groupby("tour")["is_new_album_song"].mean()

tour
Fearless                   0.633017
Speak Now World Tour       0.484631
The 1989 World Tour        0.470746
The Eras Tour              0.125174
The Red Tour               0.432911
reputation Stadium Tour    0.430544
Name: is_new_album_song, dtype: float64

In [100]:
tour_df.to_csv("../data/taylor_swift_tour_df_labeled.csv", index=False)